# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


In [ ]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given: puzzles here are not labeled easy/medium/hard. The supplied solution is only for checking your work; the solver functions must answer from `givens`.


In [ ]:
# do not change this function; it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])


5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
[3, '.', '.', '.', '.', '.', '.', '.', '.']
['.', 2, '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


## Define Symbols

Two families of propositional symbols, for row `r`, column `c`, value `v`: `Is_rcv` means cell `(r,c)` has value `v`; `Not_rcv` means that value has been eliminated from that cell.

For each representation in Task 1, determine which of these two families is needed.


### Helper function

`atom(prefix, r, c, v)` is provided in `sudoku_solver.py`; do not change it. In code, examples are `Is3_2_4` and `Not3_2_4`.


In [ ]:
# atom() is provided in sudoku_solver.py and imported above.

## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell has at least one value.
- Each cell has at most one value.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- Givens hold their stated values.

Formalize these in a general `PropKB` and a definite `PropDefiniteKB`.


In [ ]:
# Smoke-test both knowledge-base builders on the working puzzle.
general_kb = build_general_kb(n, box_h, box_w, givens)
definite_kb = build_definite_kb(n, box_h, box_w, givens)
print(f'General KB clauses: {len(general_kb.clauses)}')
print(f'Definite KB clauses: {len(definite_kb.clauses)}')
print('Given-cell query (2,1)=2 via forward chaining:',
      pl_fc_entails(definite_kb, atom('Is', 2, 1, 2)))


General KB clauses: 11775
Definite KB clauses: 23358
Given-cell query (2,1)=2 via forward chaining: True


### A.2) Solve the Puzzle

**(a) Resolution and model checking.** The full general representation is deliberately impractical for these algorithms; a small experiment is enough to observe the growth.

**(b)** Implemented `solve_full_grid_fc`.

**(c)** Implemented `pl_bc_entails`.

**(d)** Implemented `solve_full_grid_bc` and timed both full-grid solvers.


### Attempted resolution/model-checking observations

For the working puzzle, the general KB contains 11,775 clauses. Attempting unrestricted resolution on the full KB exhausted the available execution resources in this environment rather than returning. Truth-table entailment is even less tractable because the representation contains 729 Boolean `Is_rcv` atoms, implying a worst-case model space of `2^729` assignments.


### Observation notes

The general CNF representation is manageable to build, but its inference closure is not. Resolution can create a very large number of resolvents, while truth-table checking scales exponentially in the number of propositional symbols. The Horn representation is much better suited to the propagation algorithms requested by the assignment.


**(b) Forward chaining on the full grid.** `solve_full_grid_fc()` uses the definite KB and computes its Horn closure once for the full grid.

**(c) Backward chaining.** `pl_bc_entails()` uses goal-directed rule dependency expansion followed by fixed-point evaluation of only the rules reachable from the query.

**(d) Backward chaining on the full grid.** `solve_full_grid_bc()` checks each cell/value candidate with `pl_bc_entails`.


### Verifying the algorithms

The following check verifies the full-grid results, all 81 correct backward-chaining values, and a representative false query for every cell. The full-grid outputs are compared exactly with the supplied solution.


In [ ]:
import time

t0 = time.time()
solved = solve_full_grid_fc(n, box_h, box_w, givens)
fc_time = time.time() - t0
assert solved == puzzle['solution']

definite_kb = build_definite_kb(n, box_h, box_w, givens)

for (r, c), v in puzzle['solution'].items():
    assert pl_bc_entails(definite_kb, atom('Is', r, c, v)) is True

for (r, c), v in puzzle['solution'].items():
    wrong_v = 1 if v != 1 else 2
    assert pl_bc_entails(definite_kb, atom('Is', r, c, wrong_v)) is False

t0 = time.time()
solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
bc_time = time.time() - t0
assert solved_bc == puzzle['solution']

print(f'solve_full_grid_fc: {fc_time:.3f}s')
print(f'solve_full_grid_bc: {bc_time:.3f}s')
print('Both solvers exactly match the supplied solution.')
print('Backward-chaining completeness: all 81 correct cell values proved.')
print('Backward-chaining soundness: one incorrect candidate rejected for every cell.')

solve_full_grid_fc: 0.538s
solve_full_grid_bc: 29.045s
Both solvers exactly match the supplied solution.
Backward-chaining completeness: all 81 correct cell values proved.
Backward-chaining soundness: one incorrect candidate rejected for every cell.


## Part B: Conceptual Questions

### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

**Your answer:**

**(a) General KB.** I use one propositional atom `Is_rcv` to mean that cell `(r,c)` contains value `v`. Each cell gets one at-least-one clause, `(Is_rc1 | ... | Is_rcn)`. At-most-one is encoded pairwise as `(~Is_rcv1 | ~Is_rcv2)`. Row, column, and box uniqueness are encoded as pairwise “not both” clauses for each unit and value. Each given is inserted as a unit clause `Is_rcv`. The resulting `PropKB` is therefore a CNF representation of the Sudoku constraints.

**(b) Definite/Horn KB.** Horn clauses cannot directly contain the positive disjunction for “at least one value”, so I use both `Is` and `Not` as positive proposition symbols. From `Is_rcv`, rules eliminate other values from the same cell and eliminate `v` from every row/column/box peer. Last-candidate rules derive `Is_rcv` when every other value has been eliminated from that cell. Hidden-single rules derive `Is_rcv` when every other cell in a row, column, or box has `Not_rcv`. Thus the general Sudoku constraints are represented operationally using Horn-style implications rather than unrestricted disjunctions.

### 2. Theoretical Completeness vs. Computational Tractability

**Your answer:**

I would not use full truth-table model checking or unrestricted resolution as the default solver for a 9x9 Sudoku. Model checking can require `2^m` assignments for `m` Boolean variables; this encoding has 729 `Is_rcv` variables, so exhaustive enumeration is infeasible. Resolution is also sound and complete, but it can generate a huge number of resolvents as clauses interact. In my experiment the general KB had 11,775 clauses and the full resolution attempt exhausted the available runtime/memory. Horn forward/backward chaining takes advantage of the structured rule form and avoids those two forms of state-space explosion.

### 3. Backward Chaining: Design, Pseudocode, and Challenges

**Your answer:**

```text
BC-ENTAILS(KB, query):
    collect facts and rules indexed by conclusion
    expand only goals that can be reached backward from query
    for each matching rule, add all of its premises as new goals
    evaluate the finite relevant rule graph until no new fact can be proved
    return whether query is in the derived set
```

For one rule, all premises must be proved (AND). For multiple rules with the same conclusion, any one complete rule is sufficient (OR). The main correctness problems are repeated subgoals and cycles. The implementation stores visited goals/rules so a cycle such as `A -> B -> A` cannot recurse forever. Because the vocabulary is finite and the relevant dependency graph is finite, the fixed-point phase terminates.

### 4. Expressive Limits of Horn Logic

**Your answer:**

A Naked Pair can be encoded with the `Not` vocabulary. Suppose `(1,2)` and `(1,5)` are restricted to `{2,7}`. The premises can state that each of those two cells is `Not` every value other than 2 or 7. Then, for every other cell `(1,c)` in the row, add two definite rules: `all_pair_premises ==> Not_1_c_2` and `all_pair_premises ==> Not_1_c_7`. This removes 2 and 7 from the rest of the row. The consequence is a larger KB because many possible pair positions/candidate sets must be instantiated, which can strengthen deterministic propagation but can also increase indexing and search cost.

### 5. Data-Driven vs. Goal-Driven Performance

**Your answer:**

**(a)** Backward chaining is attractive when the KB is large but the query depends on a small subset. It follows only rules whose conclusions can help prove the requested cell/value, while forward chaining may derive many unrelated consequences.

**(b)** Backward chaining loses that advantage when the query touches much of the KB, when there are many alternative proof paths, or when many different queries are asked from the same KB. Then a single shared forward closure can be cheaper.

For the working 30-given puzzle in this runtime, the full-grid timings were about **0.54 s forward chaining** and **29.05 s backward chaining**. The exact values are environment-dependent, but the large gap illustrates the data-driven versus goal-driven trade-off for many queries.

## Part C: Streamlit Integration

The application in `sudoku_app.py` implements puzzle selection, a visual board, forward/backward full-grid solving with timings, a targeted entailment query, and a human-readable tutor trace. Core solver functions remain in `sudoku_solver.py`.


## Submission

Submit the completed notebook, `sudoku_solver.py`, and `sudoku_app.py` as specified by the assignment.

### Deployed Streamlit app URL

The application is implemented and tested locally. A public Streamlit Community Cloud URL requires a GitHub repository and authenticated external deployment, which is outside this execution environment.

Local command: `streamlit run sudoku_app.py`

After deployment, paste the generated `https://<name>.streamlit.app` URL into this section.
